# Lab MLP experiments

Encode the 50 lab percentile values and their 50 observation indicators supplied by our DataLoader.

Input: `(B, 100)`, with percentile values first and observation indicators second. An indicator of `1` means observed; `0` means missing. Missing values have already been imputed during dataset preprocessing.

The encoder returns shared features `(B, 256)`, distinct features `(B, 256)`, and auxiliary disease probabilities `(B, 5)`.


In [4]:
import torch
from torch import nn

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


## Lab encoder



In [5]:
class LabsMLP(nn.Module):
    def __init__(self, input_size=100, num_classes=5,
                 d_model=256, dropout=0.3):
        super().__init__()
        self.d_model = d_model

        # Encode the lab values together with their observation indicators.
        self.model_feat = nn.Sequential(
            nn.Linear(input_size, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # Shared representation branch.
        self.model_shared = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # Distinct representation branch.
        self.model_distinct = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # Auxiliary disease prediction from the distinct representation.
        self.fc_distinct = nn.Linear(d_model, num_classes)

    def forward(self, x):
        # x: (B, 100), already concatenated by the DataLoader.
        feat = self.model_feat(x)
        rep_shared = self.model_shared(feat)
        rep_distinct = self.model_distinct(feat)
        pred_distinct = self.fc_distinct(rep_distinct).sigmoid()

        return rep_shared, rep_distinct, pred_distinct


## Check the forward pass




In [6]:
model = LabsMLP().to(device)
model.eval()

# Match the DataLoader: 50 percentile values followed by 50 indicators.
labs_percentiles = torch.rand(1, 50, device=device)
labs_observed = torch.randint(0, 2, (1, 50), device=device).float()
labs = torch.cat([labs_percentiles, labs_observed], dim=1)

with torch.no_grad():
    rep_shared, rep_distinct, pred_distinct = model(labs)

print("Input shape:", labs.shape)
print("Shared representation:", rep_shared.shape)
print("Distinct representation:", rep_distinct.shape)
print("Disease predictions:", pred_distinct.shape)
print("\nPrediction probabilities:")
print(pred_distinct.cpu())


Input shape: torch.Size([1, 100])
Shared representation: torch.Size([1, 256])
Distinct representation: torch.Size([1, 256])
Disease predictions: torch.Size([1, 5])

Prediction probabilities:
tensor([[0.4991, 0.5273, 0.4959, 0.5007, 0.5116]])
